In [ ]:
def main(datasources, start_date, end_date):
    """
    因子：研发资本化（R&D Capital / Total Assets）

    文献依据：
        Lev & Sougiannis (1996) - The Capitalization, Amortization, and
        Value-Relevance of R&D, Journal of Accounting and Economics.
        核心：研发投入形成的知识资本有累积效应，对过去多期研发费用做
        指数衰减加总得到研发资本存量（RDC），再除以总资产评估强度。
        衰减系数：当期1.0，上期0.8，依次递减至第4期0.2。
        中证1000偏成长/科技属性，研发资本积累在此股票池尤为有效。

    公式：RDC / Total_Assets_LF
          RDC = 1.0*RD_t + 0.8*RD_{t-1} + 0.6*RD_{t-2}
              + 0.4*RD_{t-3} + 0.2*RD_{t-4}  (TTM)
    方向：正向（研发资本存量越高 → 预期超额收益越高）
    本地评估（2024年）：多空Sharpe=1.944，压力期IC IR=0.222
    """
    import pandas as pd
    import numpy as np
    import dai

    financial = datasources["financial"]

    # 查研发费用各期（shift 0~4 对应当期及过去4期）
    sql_rd = f"""
    SELECT date, instrument, shift,
           research_and_development_expense AS rd
    FROM {financial}
    WHERE category = 'ttm'
      AND shift IN (0, 1, 2, 3, 4)
      AND research_and_development_expense IS NOT NULL
      AND research_and_development_expense > 0
    """
    df_rd = dai.query(sql_rd, filters={"date": ["2015-01-01", end_date]}).df()
    df_rd["date"]  = pd.to_datetime(df_rd["date"])
    df_rd["rd"]    = pd.to_numeric(df_rd["rd"], errors="coerce")
    df_rd["shift"] = pd.to_numeric(df_rd["shift"], errors="coerce").astype(int)
    df_rd = df_rd.dropna(subset=["rd"])

    # 总资产单独从 lf 类别查（只有 shift=0 才有总资产）
    sql_assets = f"""
    SELECT date, instrument, total_assets AS assets
    FROM {financial}
    WHERE category = 'lf'
      AND shift = 0
      AND total_assets IS NOT NULL
      AND total_assets > 0
    """
    df_assets = dai.query(sql_assets, filters={"date": ["2015-01-01", end_date]}).df()
    df_assets["date"]   = pd.to_datetime(df_assets["date"])
    df_assets["assets"] = pd.to_numeric(df_assets["assets"], errors="coerce")
    df_assets = df_assets.dropna(subset=["assets"])
    df_assets = df_assets.drop_duplicates(subset=["date", "instrument"])

    # 透视研发费用为宽表，计算加权研发资本存量
    decay = {0: 1.0, 1: 0.8, 2: 0.6, 3: 0.4, 4: 0.2}
    wide_rd = df_rd.pivot_table(
        index=["date", "instrument"],
        columns="shift",
        values="rd",
        aggfunc="first"
    )
    wide_rd.columns = [f"rd_{int(c)}" for c in wide_rd.columns]

    rdc = pd.Series(0.0, index=wide_rd.index)
    for s, w in decay.items():
        col = f"rd_{s}"
        if col in wide_rd.columns:
            rdc += w * wide_rd[col].fillna(0)

    rdc = rdc[rdc > 0].rename("rdc").reset_index()
    rdc = pd.merge(rdc, df_assets, on=["date", "instrument"], how="inner")
    rdc["factor"] = (rdc["rdc"] / (rdc["assets"].abs() + 1.0)).clip(0, 1)
    rdc = rdc[["date", "instrument", "factor"]].dropna(subset=["factor"])

    # 公告日 → 自然日 forward-fill
    data_start = rdc["date"].min()
    natural_dates = pd.date_range(start=data_start, end=end_date)
    wide = (
        rdc.set_index(["date", "instrument"])["factor"]
        .unstack(level="instrument")
        .reindex(natural_dates)
        .ffill()
    )
    try:
        long = wide.stack(future_stack=True).reset_index()
    except TypeError:
        long = wide.stack().reset_index()
    long.columns = ["date", "instrument", "factor"]
    long["date"] = pd.to_datetime(long["date"])
    long = long.dropna(subset=["factor"])

    # 对齐中证1000成分股
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"])

    result = pd.merge(long, stk_pool, how="inner", on=["date", "instrument"])
    return result[["date", "instrument", "factor"]].reset_index(drop=True)


if __name__ == "__main__":
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    datasources = {
        "bar1m":     "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }
    start_date = "2024-01-01 00:00:00"
    end_date   = "2024-12-31 23:59:59"

    logger.info(f"计算因子，区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)
    logger.info(f"因子行数：{len(factor_data)}")

    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        start_date=start_date,
        end_date=end_date,
        process_pools=False,
        show=True,
    )
